In [0]:
from pyspark.sql.functions import *

In [0]:
df_conv = (
    spark.read.table('marketing_analytics_project.`02_silver`.fact_conversions')
    .groupBy('campaign_id', 'date')
    .agg(
        sum('revenue_usd').alias('revenue_usd'),
        sum('add_to_cart').alias('add_to_cart'),
        sum('wishlist').alias('wishlist'),
        sum('purchase').alias('purchase'),
        sum(col('add_to_cart') + col('wishlist') + col('purchase')).alias('total_no_of_conv')
    )
    .withColumn('revenue_usd', round(col('revenue_usd'), 2))
)

df_ad_perf = (
    spark.read.table('marketing_analytics_project.`02_silver`.fact_ad_performance')
    .groupBy('campaign_id', 'date')
    .agg(
        sum('impressions').alias('impressions'),
        sum('clicks').alias('clicks'),
        sum('spend_usd').alias('spend_usd')
    )
    .withColumn('spend_usd', round(col('spend_usd'), 2))
)

df_join = (
    df_ad_perf
    .join(df_conv, ['campaign_id', 'date'], 'full_outer')
    .fillna(0)
    .fillna('NA')
)

df_camp = spark.read.table('marketing_analytics_project.`02_silver`.dim_campaign')

df_final = (
    df_join
    .join(df_camp, 'campaign_id', 'left')
    .select("date", "campaign_id", "campaign_name", "channel", "impressions", "clicks", "spend_usd", "revenue_usd", "add_to_cart", "wishlist", "purchase", "total_no_of_conv")
    .orderBy('date', 'campaign_id')
)

display(df_final)


date,campaign_id,campaign_name,channel,impressions,clicks,spend_usd,revenue_usd,add_to_cart,wishlist,purchase,total_no_of_conv
2024-06-03,CMP001,Summer Sale Blast,YouTube,40343,586,108.61,129.0,0,0,1,1
2024-06-04,CMP001,Summer Sale Blast,YouTube,18016,745,554.32,79.99,2,0,1,3
2024-06-05,CMP001,Summer Sale Blast,YouTube,47086,620,270.45,0.0,2,1,0,3
2024-06-08,CMP001,Summer Sale Blast,YouTube,42676,963,544.02,0.0,1,1,0,2
2024-06-09,CMP001,Summer Sale Blast,YouTube,52583,2420,1600.3,59.99,1,0,1,2
2024-06-10,CMP001,Summer Sale Blast,YouTube,57209,1335,842.01,59.99,2,1,1,4
2024-06-10,CMP004,Weekend Flash Deals,Facebook,56989,2663,1996.99,0.0,2,1,0,3
2024-06-12,CMP001,Summer Sale Blast,YouTube,27234,864,423.61,0.0,0,0,0,0
2024-06-13,CMP004,Weekend Flash Deals,Facebook,33484,897,591.44,0.0,1,0,0,1
2024-06-14,CMP001,Summer Sale Blast,YouTube,55164,1278,671.51,0.0,1,2,0,3


In [0]:
df_conv = (
    spark.read.table('marketing_analytics_project.`02_silver`.fact_conversions')
    .groupBy('customer_id')
    .agg(
        sum('revenue_usd').alias('revenue_usd'),
        sum('add_to_cart').alias('add_to_cart'),
        sum('wishlist').alias('wishlist'),
        sum('purchase').alias('purchase'),
        sum(col('add_to_cart') + col('wishlist') + col('purchase')).alias('total_no_of_conv')
    )
    .withColumn('revenue_usd', round(col('revenue_usd'), 2))
)

df_cust = spark.read.table('marketing_analytics_project.`02_silver`.dim_customer')

df_join = (
    df_conv
    .join(df_cust, 'customer_id', 'left')
    .withColumn('customer_tier', expr("CASE WHEN revenue_usd < 100 OR total_no_of_conv < 10 THEN 'Bronze' WHEN revenue_usd >= 500 OR total_no_of_conv >= 15 THEN 'Gold' ELSE 'Silver' END"))
    .select('customer_id', 'region', 'customer_tier', 'revenue_usd', "add_to_cart", "wishlist", "purchase", "total_no_of_conv")
    .orderBy('customer_id')
)

display(df_join)

customer_id,region,customer_tier,revenue_usd,add_to_cart,wishlist,purchase,total_no_of_conv
CU501,US-West,Bronze,149.0,3,3,1,7
CU502,US-East,Silver,348.95,4,1,6,11
CU503,US-West,Silver,458.94,4,2,7,13
CU504,US-West,Silver,279.97,4,6,3,13
CU505,US-West,Silver,409.95,3,3,5,11
CU506,US-East,Silver,249.96,5,5,4,14
CU507,EU-Central,Bronze,298.0,2,3,2,7
CU508,US-East,Bronze,239.96,2,3,4,9
CU509,EU-Central,Silver,239.97,4,3,3,10
CU510,IN-North,Gold,348.95,6,4,6,16
